In [19]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy

from src.data_loader import get_data, load_raw_data
from src.features import enrich_dataframe
from scipy.stats import pearsonr

data = get_data(raw_dir="../data/raw", processed_path="../data/processed/merged_orders.parquet")
data = enrich_dataframe(data)

raw = load_raw_data(raw_dir="../data/raw")
products = raw["products"]
departments = raw["departments"]
aisles = raw["aisles"]

Chargement depuis le cache .parquet


Question : Combien de produits uniques existent par rayon/département ?

Pour répondre à cette question on va utiliser la table fusionnée data en ajoutant un .nunique() pour ne plus avoir de doublons.

Le projet fusionne les 5 datasets en une seule table (data), construite avec une jointure left depuis order_products. Cette table répond bien aux questions de comportement d'achat réel (qui a commandé quoi, quand) et donc à la majorité des nos questions, puisqu'une ligne y représente un produit effectivement présent dans une commande.

La limite de cette table pour la question présente : les produits du catalogue qui n'ont jamais été commandés par personne n'apparaissent pas dans data, puisque la jointure part de order_products.

En comparant le comptage de produits uniques par rayon sur data (produits commandés) vs sur products (catalogue complet), on observe une différence :
Exemple : Rayon "candy chocolate → 1219 produits commandés vs 1246 dans le catalogue.

Décision : pour répondre à "combien de produits existent par rayon/département", on utilise donc products (fusionnée avec aisles/departments), qui reflète le catalogue complet, plutôt que data, qui ne reflète que les produits effectivement vendus.

In [4]:
raw = load_raw_data(raw_dir="../data/raw")
products = raw["products"]
aisles = raw["aisles"]

products_with_aisle = products.merge(aisles, on="aisle_id", how="left")
products_per_aisle = products_with_aisle.groupby("aisle")["product_id"].nunique().sort_values(ascending=False)
products_per_aisle.head(10)

aisle
missing                 1258
candy chocolate         1246
ice cream ice           1091
vitamins supplements    1038
yogurt                  1026
chips pretzels           989
tea                      894
packaged cheese          891
frozen meals             880
cookies cakes            874
Name: product_id, dtype: int64

## Investigation préalable : les catégories "missing" et "other"

Avant de répondre à la question une investigation a été nécessaire suite à la découverte de deux catégories suspectes dans les données.

### 1. Découverte de "missing"

En calculant le nombre de produits uniques par rayon, la catégorie 
"missing" ressort en tête du classement.

In [4]:
missing_products_count = products[products["aisle_id"] == 100].shape[0]
missing_products_count

1258

1258 produits (~2,5% du catalogue de 49 688 produits) sont classés dans le rayon "missing" un poids important qui ne peut pas être ignoré.

### 2. Vérification systématique : d'autres catégories suspectes ?

On vérifie également le nombre de rayons distincts par département.

In [5]:
products_full = products.merge(aisles, on="aisle_id", how="left")
products_full = products_full.merge(departments, on="department_id", how="left")

aisles_per_department = products_full.groupby("department")["aisle"].nunique().sort_values()
aisles_per_department

department
other               1
missing             1
bulk                2
pets                2
breakfast           4
international       4
babies              4
canned goods        5
dry goods pasta     5
deli                5
bakery              5
alcohol             5
produce             5
meat seafood        7
beverages           8
dairy eggs         10
household          10
frozen             11
snacks             11
pantry             12
personal care      17
Name: aisle, dtype: int64

Seuls "missing" et "other" ont exactement 1 seul rayon associé à leur département une structure anormale. 

(Note : en parcourant la liste complète des rayons, deux autres noms génériques avaient été repérés comme potentiellement suspects "refrigerated" et "more household". 
Une fois vérifiés avec cette même méthode, ils se sont révélés être de vrais rayons légitimes, chacun accompagné de plusieurs autres rayons dans leur département.
Par exemple "refrigerated" appartient au département "beverages", aux côtés de tea, coffee, soft drinks...)

### 3. "other" confirmée via le treemap

La visualisation en treemap (département → rayon) a permis de repérer visuellement que le département "other" ne contient qu'un unique sous-rayon, lui aussi nommé "other" (548 produits) cohérent avec la vérification systématique ci-dessus.

### 4. Impact des deux catégories sur `data`

On a vérifié ensuite si ces deux catégories, une fois les produits 
effectivement commandés pris en compte (table `data`), représentent un poids significatif ou négligeable.

In [6]:
print("Part de 'missing' dans data :", data["aisle"].value_counts(normalize=True).get("missing", 0) * 100, "%")
print("Part de 'other' dans data :", data["aisle"].value_counts(normalize=True).get("other", 0) * 100, "%")

Part de 'missing' dans data : 0.21156603194063148 %
Part de 'other' dans data : 0.11053736114378462 %


Poids négligeable côté `data` (0,21% et 0,11%). 
On vérifie maintenant combien d'utilisateurs seraient affectés par une éventuelle exclusion.

In [8]:
users_with_missing = data[data["aisle"] == "missing"]["user_id"].unique()
only_missing = data[data["user_id"].isin(users_with_missing)].groupby("user_id")["aisle"].apply(lambda x: (x == "missing").all())
print("Utilisateurs avec au moins un produit 'missing' :", len(users_with_missing))
print("Utilisateurs avec uniquement des produits 'missing' :", only_missing.sum())

Utilisateurs avec au moins un produit 'missing' : 13468
Utilisateurs avec uniquement des produits 'missing' : 4


In [9]:
users_with_other = data[data["aisle"] == "other"]["user_id"].unique()
only_other = data[data["user_id"].isin(users_with_other)].groupby("user_id")["aisle"].apply(lambda x: (x == "other").all())
print("Utilisateurs avec au moins un produit 'other' :", len(users_with_other))
print("Utilisateurs avec uniquement des produits 'other' :", only_other.sum())

Utilisateurs avec au moins un produit 'other' : 7861
Utilisateurs avec uniquement des produits 'other' : 6


- "missing" : 13 468 utilisateurs concernés, dont seulement 4 
  exclusivement
- "other" : 7 861 utilisateurs concernés, dont seulement 6 exclusivement

### 5. Les produits eux-mêmes sont-ils fiables ?

Dernière vérification : les produits classés "missing"/"other" ont-ils un nom de produit correctement renseigné ?

In [10]:
other_aisle_id = aisles[aisles["aisle"] == "other"]["aisle_id"].values[0]
products[products["aisle_id"] == other_aisle_id][["product_id", "product_name"]].head(10)

,product_id,product_name
85,86,"Camilia, Single Liquid Doses"
505,506,Arrowroot Powder
534,535,Rescue Remedy
675,676,Sunset Blush
712,713,Pink Moscato California
1009,1010,Agar Agar Powder
1012,1013,Poly-Vi-Sol® Infants And Toddlers Multivitamin...
1654,1655,Moderate Absorbency for Women Underwear Large
1681,1682,Pink Moscato
1723,1724,Coconut Lime Frozen Pop Dairy Free Gluten Free


Les produits "other" ont des noms parfaitement renseignés (ex : "Rescue Remedy", "Pink Moscato California"). 
Vérification pour "missing" :

In [11]:
missing_aisle_id = aisles[aisles["aisle"] == "missing"]["aisle_id"].values[0]
products[products["aisle_id"] == missing_aisle_id][["product_id", "product_name"]].head(10)

,product_id,product_name
37,38,Ultra Antibacterial Dish Liquid
71,72,Organic Honeycrisp Apples
109,110,Uncured Turkey Bologna
296,297,"Write Bros Ball Point Pens, Cap-Pen, Medium (1..."
416,417,Classics Baby Binks Easter Chocolate Bunny
436,437,Strawberry Cheesecake Nonfat Yogurt
439,440,Crossovers Olive Thyme Almond
471,472,Fancy Diced Hot Chilies
490,491,Pompelmo Water
555,556,Bake & Break Cheese Garlic Loaf


Même constat pour "missing" (ex : "Organic Honeycrisp Apples", "Ultra Antibacterial Dish Liquid"). Les deux catégories regroupent des produits sans lien thématique et ce n'est pas d'un problème de fiabilité sur les produits.

### 6. Décision finale

La table `data` est conservée telle quelle, sans filtrage. Le 
problème identifié ne concerne que la catégorisation rayon/département de certains produits  pas la fiabilité de l'information produit ou de commande elle-même. 
Exclure ces lignes ferait perdre des données de vente légitimes, utiles pour toute question ne portant pas spécifiquement 
sur les rayons/départements.

Cette décision ne concerne que l'analyse Rayon/Département : chaque graphique de cette section identifiera visuellement "missing" et "other" (couleur distincte) plutôt que de les exclure, pour rester transparent sur leur présence sans fausser la lecture des vrais rayons.

### Visualisation : répartition des produits par département et rayon

Le treemap ci-dessous représente tous les départements et rayons du catalogue, y compris "missing" et "other" (voir investigation ci-dessus pour leur origine).

In [12]:
raw = load_raw_data(raw_dir="../data/raw")
products = raw["products"]
aisles = raw["aisles"]
departments = raw["departments"]

products_full = products.merge(aisles, on="aisle_id", how="left")
products_full = products_full.merge(departments, on="department_id", how="left")

In [13]:
fig = px.treemap(
    products_full,
    path=["department", "aisle"],
    title="Répartition des produits uniques par département puis par rayon",
)

fig.update_traces(textinfo="label+value")
fig.show()

**Réponse à la question :** 
le nombre de produits uniques référencés varie fortement d'un rayon et d'un département à l'autre. Au niveau département, "personal care" (17 rayons), "snacks", "pantry" et "beverages" sont les plus fournis en diversité de rayons, tandis que des départements comme "pets", "meat seafood" ou "deli" n'en comptent que 2 à 5.

Au niveau rayon, "candy chocolate" (1246 produits) domine largement, suivi de "vitamins supplements" (1038) et "hair care" (816) contre certains rayons très restreints comme "meat seafood" ou "pets", qui n'affichent que quelques dizaines de produits.

**Implication business :** 
cette forte disparité soulève la question de savoir si elle reflète une vraie différence de demande client, ou si certains rayons sont structurellement sous-fournis par rapport à leur potentiel de vente. 
C'est précisément ce que la question suivante (diversité de produits vs volume de commandes) va permettre de vérifier.

## Question 2 : Diversité de produits et volume de commandes

**Question :** 
Y a-t-il un lien entre la diversité de produits d'un rayon et son volume de commandes ?

**Hypothèse à tester :** 
Il existe une corrélation entre la diversité de produits référencés dans un rayon et son volume de commandes.

### Pourquoi exclure "missing" et "other" de ce calcul

L'investigation précédente a montré que "missing" et "other" sont des catégories techniques de repli, regroupant des produits sans lien thématique entre eux (dentifrice, vin, papeterie...),contrairement aux vrais rayons qui rassemblent des produits d'une même famille cohérente.

Une corrélation mesure si deux variables suivent une même logique d'évolution. Inclure "missing"/"other" reviendrait à mélanger, dans le calcul, deux points de données qui ne suivent pas la même dynamique qu'un vrai rayon (leur diversité et leur volume ne reflètent pas un choix d'assortiment cohérent, juste un regroupement de produits disparates). Cela risquerait de fausser le coefficient de corrélation, en particulier parce que "missing" (1258 produits) est un point extrême qui pourrait influencer le résultat de façon disproportionnée.

Ces deux catégories restent néanmoins documentées et visibles ailleurs dans l'analyse (treemap, section investigation) elles ne sont exclues que localement, pour ce test statistique précis.

On construit deux mesures par rayon :
- `product_diversity` : nombre de produits uniques référencés (à partir du catalogue `products`)
- `order_volume` : nombre de fois où un produit de ce rayon a été 
  commandé (à partir de `data`, qui reflète le comportement d'achat réel)

In [15]:
diversity_per_aisle = products_full[products_full["aisle"] != "other"]
diversity_per_aisle = diversity_per_aisle[diversity_per_aisle["aisle"] != "missing"]
diversity_per_aisle = diversity_per_aisle.groupby("aisle")["product_id"].nunique()
diversity_per_aisle.name = "product_diversity"

volume_per_aisle = data[~data["aisle"].isin(["missing", "other"])]
volume_per_aisle = volume_per_aisle.groupby("aisle")["product_id"].count()
volume_per_aisle.name = "order_volume"

aisle_stats = pd.concat([diversity_per_aisle, volume_per_aisle], axis=1).reset_index()
aisle_stats.head()

,aisle,product_diversity,order_volume
0,air fresheners candles,355,6529
1,asian foods,605,55247
2,baby accessories,44,2672
3,baby bath body care,132,2802
4,baby food formula,718,127449


Pour tester notre hypothèse, on utilise `scipy.stats`, la librairie de référence en Python pour les calculs statistiques (tests d'hypothèses, distributions de probabilité, corrélations...). On calcule ici le coefficient de corrélation de Pearson entre nos deux variables.

In [20]:
correlation, p_value = pearsonr(aisle_stats["product_diversity"], aisle_stats["order_volume"])
print(f"Coefficient de corrélation de Pearson : {correlation:.3f}")
print(f"P-value : {p_value:.5f}")

Coefficient de corrélation de Pearson : 0.299
P-value : 0.00051


**Résultat du test statistique :**
- Coefficient de corrélation de Pearson : 0,299
- P-value : 0,00051 (< 0,05, donc statistiquement significatif)

**Interprétation :** il existe bien une corrélation positive entre la diversité de produits d'un rayon et son volume de commandes, et cette relation est statistiquement significative (peu probable qu'elle soit due au hasard). 
Cependant, la force de cette corrélation reste faible à modérée (0,299 sur une échelle de -1 à +1) : la diversité de produits n'explique qu'une partie limitée des écarts de volume entre rayons. 
D'autres facteurs (emplacement en magasin, prix, habitudes d'achat, fréquence de consommation du type de produit) jouent probablement un rôle au moins aussi important.

**Implication business :** élargir l'assortiment d'un rayon pourrait avoir un effet positif sur ses ventes, mais ce levier seul ne suffira probablement pas à expliquer ou corriger de grands écarts de performance entre rayons il faudrait le combiner à d'autres actions (mise en avant, emplacement en magasin).

In [22]:
fig = px.scatter(
    aisle_stats,
    x="product_diversity",
    y="order_volume",
    hover_name="aisle",
    title="Diversité de produits vs volume de commandes, par rayon",
    labels={"product_diversity": "Nombre de produits uniques", "order_volume": "Volume de commandes"},
    trendline="ols",
)
fig.show()

Le nuage de points confirme visuellement la corrélation faible mesurée la tendance générale est positive (ligne de tendance croissante), mais la majorité des rayons se concentrent avec  un volume de commandes modéré, quelle que soit leur diversité.

Deux rayons se démarquent nettement avec un volume de commandes très 
supérieur au reste (au-delà d'1 million de commandes), pour une 
diversité de produits pourtant modérée (environ 400 à 600 produits). 
Ces rayons à très fort volume (des produits de consommation courante et récurrente, comme les fruits/légumes frais) illustrent qu'un volume élevé de commandes peut s'obtenir sans nécessiter un large assortiment ce qui explique en partie la faiblesse de la corrélation globale.

**Conclusion :** 
la diversité de produits joue un rôle réel mais limité dans le volume de commandes d'un rayon. D'autres facteurs comme la 
fréquence de consommation naturelle du type de produit (produits 
essentiels vs produits occasionnels) semblent avoir un poids au moins aussi important.

## Question 3 : Diversité de panier et segment client

**Question :** 
Dans un panier, combien de produits de différents départements/rayons peut-on trouver, et cette diversité diffère-t-elle selon le segment client ?

**Hypothèse à tester :** 
On suppose qu'un client plus fidèle (ayant 
commandé plus fréquemment) a statistiquement plus d'occasions d'explorer le site, et pourrait donc présenter une diversité de 
panier plus élevée que les clients occasionnels.

**Précision méthodologique :** 
L'analyse mesure la diversité d'un panier donné(une commande individuelle) pas la diversité cumulée sur l'ensemble de l'historique d'achat d'un client. 

In [23]:
data[["order_id", "dept_per_order", "customer_segment"]].head()

,order_id,dept_per_order,customer_segment
0,2,3,typique
1,2,3,typique
2,2,3,typique
3,2,3,typique
4,2,3,typique


In [24]:
orders_diversity = data.drop_duplicates(subset="order_id")[["order_id", "dept_per_order", "customer_segment"]]
orders_diversity.shape

(1064689, 3)

In [25]:
diversity_by_segment = orders_diversity.groupby("customer_segment", observed=True)["dept_per_order"].describe()
diversity_by_segment

,count,mean,std,min,25%,50%,75%,max
customer_segment,,,,,,,,
occasionnel,141236.0,4.676187,2.594878,1.0,3.0,4.0,6.0,17.0
typique,325665.0,4.783772,2.594522,1.0,3.0,4.0,6.0,17.0
fidèle,597788.0,4.721697,2.511192,1.0,3.0,4.0,6.0,17.0


**Résultat :** 
contrairement à l'intuition, la diversité de départements 
dans un panier ne varie quasiment pas selon le segment client. Les 
statistiques sont remarquablement similaires entre les 3 groupes :


**Interprétation :** le nombre de segments client (basé sur le volume 
total de commandes) n'influence pas le comportement de diversité 
d'achat au sein d'un panier donné. Que ce soit une première commande ou 
la centième, un client explore en moyenne 4 à 5 départements différents 
par panier.

**Implication business :** la fidélisation client (faire revenir 
davantage un client) ne semble pas naturellement s'accompagner d'un 
élargissement de ses habitudes d'achat en magasin. Des actions ciblées 
(recommandations croisées, promotions inter-rayons) pourraient être 
nécessaires pour encourager les clients fidèles à explorer davantage de 
départements, plutôt que de compter sur cet effet de façon spontanée.

In [28]:
orders_diversity_aisle = data.drop_duplicates(subset="order_id")[["order_id", "aisles_per_order", "customer_segment"]]
orders_diversity_aisle.shape

(1064689, 3)

In [29]:
diversity_by_segment_aisle = orders_diversity_aisle.groupby("customer_segment", observed=True)["aisles_per_order"].describe()
diversity_by_segment_aisle

,count,mean,std,min,25%,50%,75%,max
customer_segment,,,,,,,,
occasionnel,141236.0,7.082500,4.774840,1.0,3.0,6.0,10.0,42.0
typique,325665.0,7.303742,4.832166,1.0,4.0,6.0,10.0,47.0
fidèle,597788.0,7.272013,4.707685,1.0,4.0,6.0,10.0,51.0


**Interprétation :** 
Au niveau rayon, la même conclusion se confirme la diversité par panier ne varie quasiment pas selon le segment client (moyennes et médianes très proches). Seul le maximum observé augmente avec la fidélité, ce qui s'explique simplement par le plus grand nombre de commandes des clients fidèles (plus d'opportunités d'avoir, un jour donné, un panier exceptionnellement varié) mais ce n'est pas une tendance de fond.

**Conclusion générale (départements et rayons) :** 
Le comportement de diversité d'achat par panier est stable, peu importe le segment client. Ce résultat, bien que ne confirmant pas l'intuition initiale, est un insight solide la fidélisation d'un client n'entraîne pas naturellement un élargissement de ses habitudes d'achat en magasin.

In [31]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2, subplot_titles=("Diversité par département", "Diversité par rayon"))

for segment in ["occasionnel", "typique", "fidèle"]:
    fig.add_trace(
        go.Box(y=orders_diversity[orders_diversity["customer_segment"] == segment]["dept_per_order"], name=segment),
        row=1, col=1
    )

for segment in ["occasionnel", "typique", "fidèle"]:
    fig.add_trace(
        go.Box(y=orders_diversity_aisle[orders_diversity_aisle["customer_segment"] == segment]["aisles_per_order"], name=segment, showlegend=False),
        row=1, col=2
    )

fig.update_layout(title="Diversité du panier par segment client", showlegend=False, height=500)
fig.show()

### Conclusion

**Résultat :** 
l'hypothèse initiale est infirmée. Contrairement à ce qu'on aurait pu attendre, la diversité d'un panier donné (en départements comme en rayons) est quasiment identique entre les 3 segments clients :


Le box plot confirme visuellement cette similarité les 3 boîtes se superposent presque parfaitement sur les deux niveaux de granularité. 

Seuls les maximums extrêmes (valeurs aberrantes) augmentent légèrement
avec la fidélité un effet mécanique lié au plus grand nombre de commandes des clients fidèles, pas à un vrai changement de comportement d'achat.

**Interprétation :** 
être un client fidèle (commander souvent) n'amène pas naturellement à varier davantage ses achats au sein d'un panier donné. La fidélité (fréquence/volume de commandes) et la diversité d'achat sont deux dimensions indépendantes du comportement client.

**Implication business :** 
on ne peut pas compter sur la fidélisation seule pour élargir les habitudes d'achat des clients vers plus de rayons/départements. Si le magasin souhaite encourager les clients fidèles à explorer davantage (et donc potentiellement augmenter le panier moyen), des actions ciblées seraient nécessaires  par exemple des recommandations croisées entre rayons, des promotions découverte, ou une mise en avant de produits de départements peu visités par ce client en particulier.

## Conclusion générale — Rayon/Département

Cette section visait à répondre à une question business centrale : **l'architecture du magasin (répartition des produits par rayon/ département) influence-t-elle les commandes, et faut-il mettre certains rayons en avant ?**

Les trois analyses menées convergent vers un même constat : la diversité de produits, bien qu'elle joue un rôle, n'est pas le levier principal du succès d'un rayon ou de l'exploration d'achat d'un client.

- La quantité de produits référencés varie fortement d'un rayon à l'autre (Q1), mais un large assortiment ne garantit pas mécaniquement un volume de ventes élevé : la corrélation mesurée reste faible (0,299), et certains rayons à forte rotation (produits du quotidien comme les fruits/légumes frais) génèrent un volume très élevé malgré une diversité modérée (Q2).
- La fidélisation d'un client (mesurée par son volume de commandes)n'entraîne pas naturellement une exploration plus large du magasin : un client fidèle garde le même profil de diversité de panier qu'un client occasionnel (Q3).

**Recommandation :** plutôt que d'élargir uniformément l'assortiment de 
tous les rayons en espérant un gain de ventes automatique, le magasin 
gagnerait à identifier spécifiquement les rayons à fort potentiel de 
rotation (produits de consommation courante) et à concentrer les 
efforts de mise en avant sur ceux-ci. En parallèle, des actions 
ciblées (recommandations croisées, promotions découverte) seraient 
nécessaires pour encourager activement les clients fidèles comme 
occasionnels à diversifier leurs achats, plutôt que de compter sur 
un effet naturel de la fidélisation.